<a href="https://colab.research.google.com/github/IBREEZZ/Code_Academy_Makeen2/blob/main/Day_1_Code_Academy_Students.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧠 Day 1: Hands-on with Language Models using Phi-3
Welcome to the first session of the Generative AI workshop!

Today we'll explore the basics of large language models (LLMs) and use Microsoft's **Phi-3 Mini** model.

### 🎯 Objectives
- Understand what a language model is
- Load and run a small LLM
- Generate text from prompts
- Modify prompts and analyze outputs
- Reflect on tokenization and model behavior

## 🔧 Setup the Environment

In [1]:
 %%capture
 !pip install transformers accelerate

## 📦 Load the Phi-3 Model
Fill in the missing arguments to complete the model and tokenizer loading.

In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer

# TODO: Load the model here
model = AutoModelForCausalLM.from_pretrained( #from_pretrained: Loads a pre-trained model from the Hugging Face Hub.
    "microsoft/phi-3-mini-4k-instruct", # Hint: it's a Phi-3 variant, look for the 4k instruct model --This is the Phi-3 Mini model with a 4k context window, optimized for instructions.
    device_map="auto",   # Hint: 'cuda' or 'auto' --Lets Transformers choose the best device automatically (CPU or GPU).
    torch_dtype="auto",  # Hint: dtype hint --PyTorch will pick the most compatible data type (like float16 for GPU or float32 for CPU).
    trust_remote_code=False #false:Keeps it secure — no custom code is pulled from the model repo.
)

# TODO: Load the tokenizer here
tokenizer = AutoTokenizer.from_pretrained("microsoft/phi-3-mini-4k-instruct")
#the tokenizer:
#Splits text into tokens (words or subwords).
#Converts tokens into numerical IDs that the model can understand.
#After generation, decodes those IDs back into readable text.




/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

## 📦 Create a Text Generation Pipeline

Wrap the model and tokenizer into a convenient `pipeline` object for easy inference.


In [3]:
from transformers import pipeline

# TODO: Create a pipeline for text-generation using the model and tokenizer
generator = pipeline(
    "text-generation",               # Hint: task type
    model=model,
    tokenizer=tokenizer,
    return_full_text=False,
    max_new_tokens=500,  # Hint: token cap
    do_sample=False #disables random sampling; uses greedy decoding for deterministic output.
)


Device set to use cuda:0
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


## 💬 Create and Send a Prompt

Finally, we create our prompt as a user and give it to the model:


In [4]:
# TODO: Define a user message that asks for a joke
messages = [
    {"role": "user", "content": "Tell me a funny joke about cats :"}  # Hint: Something humorous
]

# Generate output  --Calls your generator pipeline.
output = generator(messages)

# TODO: Extract and print the model’s response
print(output[0]['generated_text'])
#The pipeline returns a list of results.
#Each result is a dictionary with the key 'generated_text'.
#[0] means “give me the first result” → then print its generated text.



The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


 Why don't cats play poker in the jungle? Too many cheetahs!


## ✉️ Generate a Custom Output from a Prompt

Now let’s directly tokenize a prompt and run it through the model to generate a complete response.


In [5]:
# TODO: Write a detailed prompt that includes "<|assistant|>" at the end
prompt =  (
    "You are a helpful assistant. "
    "Oh sure, because everyone totally wants to hear yet another boring explanation of how large language models work. Make it so sarcastic that even a cat would roll its eyes."
    "<|assistant|>"
)

# Tokenize the prompt
input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to("cuda")

# TODO: Use the model to generate output from input_ids
generation_output = model.generate(
    input_ids=input_ids,
    max_new_tokens=200  # Hint: max tokens for generation, somewhere along the range of 100 to 1000
)

# TODO: Decode and print the output
print(tokenizer.decode(generation_output[0], skip_special_tokens=True))


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


You are a helpful assistant. Oh sure, because everyone totally wants to hear yet another boring explanation of how large language models work. Make it so sarcastic that even a cat would roll its eyes. Ah, the eager seeker of knowledge, ready to dive into the mystical world of large language models. Prepare yourself for a journey through the digital labyrinth, where words are conjured from the ether and sentences are spun like silk from the loom of a thousand computers.

Imagine, if you will, a colossal library, its shelves stretching into infinity, filled with every book ever written. This is the vast repository of human knowledge that our language model draws from. But unlike the librarians of old, who would painstakingly thumb through pages to find the right book, our model doesn't need to leave its cozy digital nook.

Instead, it employs a magical spell known as "machine learning." This spell allows it to learn from the patterns and structures of language, much like a child learns t

# ✍️ Reflection Prompt
## Try changing the prompt to a sarcastic tone or use specific instructions.
## What do you observe in the outputs? Discuss with your team.


## 1️⃣ View the Token IDs

After tokenizing the prompt, we can print out the list of token IDs generated by the tokenizer.


In [6]:
# TODO: Print the token IDs from our tokenized input
print(input_ids)  # Hint: What variable stores our tokenized input?

# Expected output format:
# tensor([[14359, 385, 4376, 27746, 5281, 394, 19235, 363, 278, 25305, ...]])

tensor([[  887,   526,   263,  8444, 20255, 29889,  6439,  1854, 29892,  1363,
         14332, 14909, 10753,   304,  8293,  3447,  1790,   289,  8253,  8252,
           310,   920,  2919,  4086,  4733,   664, 29889,  8561,   372,   577,
         22887,  4384,   293,   393,  1584,   263,  6635,   723,  9679,   967,
          5076, 29889, 32001]], device='cuda:0')


## 2️⃣ Decode Each Token

We can decode each token ID individually to better understand how the model splits the input into subwords.


In [7]:
# TODO: Create a loop to decode each token individually
for token_id  in input_ids[0]:  # Hint: What should we iterate over?
    print(tokenizer.decode(token_id.item()))  # Hint: What method converts token IDs back to text?

You
are
a
helpful
assistant
.
Oh
sure
,
because
everyone
totally
wants
to
hear
yet
another
b
oring
explanation
of
how
large
language
models
work
.
Make
it
so
sar
cast
ic
that
even
a
cat
would
roll
its
eyes
.
<|assistant|>


## 🧬 Inspect the Raw Model Output

The model returns a tensor of token IDs as its output. These represent the full generated sequence (input + new tokens).


In [8]:
generation_output

tensor([[  887,   526,   263,  8444, 20255, 29889,  6439,  1854, 29892,  1363,
         14332, 14909, 10753,   304,  8293,  3447,  1790,   289,  8253,  8252,
           310,   920,  2919,  4086,  4733,   664, 29889,  8561,   372,   577,
         22887,  4384,   293,   393,  1584,   263,  6635,   723,  9679,   967,
          5076, 29889, 32001,  9070, 29892,   278, 19888,  1074,  3946,   310,
          7134, 29892,  7960,   304,   270,   573,   964,   278, 16624,   936,
          3186,   310,  2919,  4086,  4733, 29889,   349,  3445,   598,  7535,
           363,   263, 16342,  1549,   278, 13436,  9775,  4316, 22653, 29892,
           988,  3838,   526,  9589,  2955,   515,   278, 29871,  1979,   322,
         25260,   526,   805,   348,   763,  4047, 29895,   515,   278,   658,
           290,   310,   263, 10405, 23226, 29889,    13,    13,  1888, 22094,
         29892,   565,   366,   674, 29892,   263,   784,  2209,   284,  3489,
         29892,   967,   528,   295,  1960, 16116,  

## 🔡 Combine Tokens into Words

Subword tokenizers may split a word into multiple pieces. You can decode them individually or as a group to see how they combine into meaningful text.


In [9]:
# TODO: Decode individual tokens to see subword splitting
print(tokenizer.decode([3323]))   # Hint: Try token ID 3323
print(tokenizer.decode([622]))   # Hint: Try token ID 622
print(tokenizer.decode([3323, 622]))  # Hint: Combine tokens
print(tokenizer.decode([29901]))   # Hint: Try token ID 29901

# Expected output pattern:
# Sub
# ject
# Subject
# .

Sub
ject
Subject
:


###🌈 Token Coloring Function

This function uses ANSI escape codes to highlight each token in a different background color. It helps visualize how text is broken down into subword units.


In [10]:
from transformers import AutoModelForCausalLM, AutoTokenizer

colors_list = [
    '102;194;165', '252;141;98', '141;160;203',
    '231;138;195', '166;216;84', '255;217;47'
]

def show_tokens(sentence, tokenizer_name):
   # TODO: Load the tokenizer from the pretrained model
   tokenizer = AutoTokenizer.from_pretrained(tokenizer_name)  # Hint: What variable contains the model name?

   # TODO: Tokenize the sentence to get token IDs
   token_ids = tokenizer(sentence).input_ids  # Hint: What text should we tokenize? --okenizes the sentence into token IDs

   for idx, t in enumerate(token_ids):
       print(
           f'\033[38;2;0;{colors_list[idx % len(colors_list)]}m' +
           tokenizer.decode([t]) +  # Hint: What method converts token ID back to text?
           '\033[0m',
           end='' # Hint: What should this end with?
       )

## 🧪 Try It on a Complex Sentence

Now test the tokenizer on a sentence with mixed content: capital letters, emojis, symbols, numbers, and spacing.


In [11]:
# TODO: Create a test sentence with mixed content
text = """
WOW! 🤯 Did you know that LLMs cost $1,000,000+ to train?! #MindBlown فلنكتشف🚀✨
"""

## 🆚 `bert-base-uncased` Tokenizer

This tokenizer lowercases all input and splits words into WordPiece subwords. Notice how it handles casing and unknown characters.


In [12]:
show_tokens(text, "bert-base-uncased")  # Hint: What text and tokenizer name?

[CLS]wow![UNK]didyouknowthatll##mscost$1,000,000+totrain?!#mind##bl##own[UNK][SEP]

## 🆚 `bert-base-cased` Tokenizer

Unlike the uncased version, this tokenizer preserves capitalization. Compare the tokenization output to see how casing influences token splitting.


In [13]:
show_tokens(text, "bert-base-cased")  # Hint: What text and tokenizer name?

[CLS]W##OW![UNK]DidyouknowthatLL##Mscost$1,000,000+totrain?!#Mind##B##lown[UNK][SEP]

## 🆚 `gpt2` Tokenizer

GPT-2 uses Byte-Pair Encoding (BPE), which often results in different token splits, especially with punctuation, emojis, or spacing.


In [14]:
show_tokens(text, "gpt2")  # Hint: What text and tokenizer name?


WOW! ��� Did you know that LLMs cost $1,000,000+ to train?! #MindBlown ��لن��ت���������


## 🆚 `google/flan-t5-small` Tokenizer

This model uses SentencePiece, which breaks down text in a more language-agnostic way. Observe how it segments common phrases and subwords.


In [15]:
show_tokens(text, "google/flan-t5-small")  # Hint: What text and tokenizer name?

WOW!<unk>DidyouknowthatLLMscost$1,000,000+totrain?!#MindBlown<unk></s>

## 🆚 `Xenova/gpt-4` Tokenizer

This Hugging Face-hosted tokenizer mirrors OpenAI's `tiktoken`. It uses Byte-Pair Encoding and handles punctuation, numbers, and special symbols distinctly.


In [16]:
# The official is `tiktoken` but this the same tokenizer on the HF platform
show_tokens(text, "Xenova/gpt-4")  # Hint: What text and tokenizer name?


WOW! ��� Did you know that LLMs cost $1,000,000+ to train?! #MindBlown فلنكتشف�����


## 🆚 `bigcode/starcoder2-15b` Tokenizer

You need access to use the actual model, but the tokenizer is available. It's optimized for code and performs differently on natural language and structured inputs.


In [17]:
# You need to request access before being able to use this tokenizer
show_tokens(text, "bigcode/starcoder2-15b")  # Hint: What text and tokenizer name?


WOW! �� Did you know that LLMs cost $1,000,000+ to train?! #MindBlown فلنكتشف�����


## 🆚 `microsoft/Phi-3-mini-4k-instruct` Tokenizer

This tokenizer is designed for compact, efficient language modeling. Notice how it splits and groups tokens differently than BERT or GPT models.


In [18]:
show_tokens(text, "microsoft/Phi-3-mini-4k-instruct")  # Hint: What text and tokenizer name?


WOW!����DidyouknowthatLLMscost$1,000,000+totrain?!#MindBlownفلنكتشف�������


## 🧠 Load a Model to Extract Embeddings

We can use a pretrained transformer (like DeBERTa) to convert text into token-level embeddings.


In [19]:
from transformers import AutoModel, AutoTokenizer

# TODO: Load a tokenizer for embeddings extraction
tokenizer = AutoTokenizer.from_pretrained("microsoft/deberta-base")  # Hint: What model name for DeBERTa-base?

# TODO: Load a language model for embeddings
model = AutoModel.from_pretrained("microsoft/deberta-v3-xsmall")  # Hint: Same model name as tokenizer

# TODO: Tokenize the sentence with proper tensor format
tokens = tokenizer('Hello My Name Is Ibreez!', return_tensors='pt')  # Hint: What text and tensor format?

# TODO: Process the tokens through the model to get embeddings
output = model(**tokens)[0]  # Hint: What variable contains our tokenized input?

## 🔢 View Embedding Dimensions

Each token is mapped to a high-dimensional vector. Let’s inspect the shape of the output to understand the model’s internal representation.


In [20]:
# TODO: Check the shape of the embedding output
output.shape  # Hint: What variable contains our model output?

# Expected output: torch.Size([1, 4, 384])
#[batch_size, sequence_length(num of token), hidden_size(embedding dimension for each token)]
#Each token is mapped to a vector with 384 values.


torch.Size([1, 10, 384])

## 🔍 Decode Input Tokens

We can decode the tokens back to their original text to understand which words each embedding vector corresponds to.


In [21]:
# TODO: Decode the input tokens back to text
for token in tokens['input_ids'][0]:  # Hint: What key contains the input IDs?
    print(tokenizer.decode([token.item()]))  # Hint: What method decodes tokens?

#.item() converts the token from a Tensor to an int
#decode converts the token ID back to the text piece


#[CLS] and [SEP] tokens → added automatically for special purposes.
#The name Ibreez was split into subwords Ib, ree, z because the WordPiece tokenizer did not find it in the vocab as a single word.
#This is normal and helps the model handle rare or unknown words.


[CLS]
Hello
 My
 Name
 Is
 Ib
ree
z
!
[SEP]


## 🧬 View Token-Level Embeddings

Each row in the tensor represents a single token’s embedding — a high-dimensional numerical representation that captures meaning and context.


In [22]:
# TODO: View the token-level embeddings tensor
output  # Hint: What variable contains our model output?

# Each row represents a single token's embedding - a high-dimensional numerical representation

tensor([[[-3.3481,  0.1979, -0.1197,  ..., -0.3541, -0.4443,  0.0331],
         [-0.4335,  0.2643,  0.3031,  ..., -0.2943,  0.0465, -1.2725],
         [-0.5504,  0.4398,  0.5564,  ..., -0.5601,  0.0606, -0.3996],
         ...,
         [ 0.0820,  0.5090,  0.0941,  ..., -0.7791,  0.1110,  0.1726],
         [-0.2175,  0.6889,  0.5124,  ..., -0.6927, -0.2799,  0.0138],
         [-3.1861,  0.1962, -0.1132,  ..., -0.1833, -0.3060,  0.4031]]],
       grad_fn=<NativeLayerNormBackward0>)

In [23]:
print(output.shape)

torch.Size([1, 10, 384])


## 🧠 Encode a Sentence into a Vector

We can convert an entire sentence into a fixed-size embedding vector using a pretrained model. This helps machines understand and compare sentences by their meaning.


In [24]:
from sentence_transformers import SentenceTransformer

# TODO: Load a sentence transformer model
model = SentenceTransformer('sentence-transformers/all-mpnet-base-v2')  # Hint: What's the model name for all-mpnet-base-v2?

# TODO: Convert text to sentence embeddings
vector = model.encode("Best movie ever!")  # Hint: What text should we encode, like "Best movie ever!"?
#Turns "Best movie ever!" into a fixed-size embedding vector.

## 📏 Check the Sentence Embedding Size

Let’s inspect the dimensions of the sentence vector. This tells us how many numerical features are used to represent the meaning of the sentence.


In [25]:
# TODO: Check the dimensions of the sentence embedding vector
vector.shape  # Hint: What variable contains our sentence vector?

# Expected output: (768,)

(768,)

## 🌐 Load Pretrained GloVe Embeddings

We can use GloVe (trained on Wikipedia and Gigaword) to explore classic word embeddings. These models capture word meaning based on co-occurrence patterns.


In [26]:
# !pip install -U numpy gensim

In [27]:
!pip install gensim
import gensim.downloader as api

# TODO: Download GloVe embeddings (50MB, trained on Wikipedia, vector size: 50)
model = api.load('glove-wiki-gigaword-50')  # Hint: What's the model name for glove-wiki-gigaword-50?

[==================================================] 100.0% 66.0/66.0MB downloaded


## 🔍 Find Similar Words in Embedding Space

We can now explore semantic similarity using vector distance. The model returns words that are closest to `"king"` in embedding space.


In [28]:
# TODO: Find words most similar to "king" in embedding space
model.most_similar(['cat'], topn=5)  # Hint: What word to search for and how many results?

# Expected output: List of (word, similarity_score) tuples

[('dog', 0.9218006134033203),
 ('rabbit', 0.8487821221351624),
 ('monkey', 0.8041081428527832),
 ('rat', 0.7891963124275208),
 ('cats', 0.7865270376205444)]

In [29]:
model.most_similar(['king'], topn=5)

[('prince', 0.8236179351806641),
 ('queen', 0.7839043140411377),
 ('ii', 0.7746230363845825),
 ('emperor', 0.7736247777938843),
 ('son', 0.766719400882721)]

In [30]:
model.most_similar(['red'], topn=5)

[('yellow', 0.8995459079742432),
 ('blue', 0.8901659250259399),
 ('green', 0.8561931848526001),
 ('black', 0.8400583863258362),
 ('purple', 0.8323202729225159)]

## 📂 Load and Parse Playlist Data

We begin by downloading a playlist dataset and parsing it into a usable format. Each playlist is represented as a sequence of song IDs.


In [31]:
import pandas as pd
from urllib import request

# TODO: Get the playlist dataset file
data = request.urlopen('https://storage.googleapis.com/maps-premium/dataset/yes_complete/train.txt')

# TODO: Parse the playlist dataset file, skipping metadata lines
lines = data.read().decode('utf-8').split('\n')[2:]  # Hint: What encoding and split character?

# TODO: Remove playlists with only one song
playlists = [s.strip().split() for s in lines if len(s.split()) > 1]  # Hint: Minimum songs per playlist?

# TODO: Load song metadata
songs_file = request.urlopen('https://storage.googleapis.com/maps-premium/dataset/yes_complete/song_hash.txt')
songs_file = songs_file.read().decode('utf-8').split('\n')  # Hint: Encoding and split character?
songs = [s.strip().split('\t') for s in songs_file]

# TODO: Create a DataFrame with song information
songs_df = pd.DataFrame(data=songs, columns=['song_id', 'artist', 'title'])  # Hint: What data and column names?
songs_df = songs_df.set_index('song_id')  # Hint: What column to use as index?

## 🎧 Display Sample Playlists

Let’s preview a couple of playlists to understand the structure. Each ID corresponds to a specific song.


In [32]:
# TODO: Display sample playlists to understand the structure
print('Playlist #1:\n ', playlists[0], '\n')  # Hint: Which playlist list and index?
print('Playlist #2:\n ', playlists[1])        # Hint: Which playlist list and index?

Playlist #1:
  ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22', '23', '24', '25', '26', '27', '28', '29', '30', '31', '32', '33', '34', '35', '36', '37', '38', '39', '40', '41', '2', '42', '43', '44', '45', '46', '47', '48', '20', '49', '8', '50', '51', '52', '53', '54', '55', '56', '57', '25', '58', '59', '60', '61', '62', '3', '63', '64', '65', '66', '46', '47', '67', '2', '48', '68', '69', '70', '57', '50', '71', '72', '53', '73', '25', '74', '59', '20', '46', '75', '76', '77', '59', '20', '43'] 

Playlist #2:
  ['78', '79', '80', '3', '62', '81', '14', '82', '48', '83', '84', '17', '85', '86', '87', '88', '74', '89', '90', '91', '4', '73', '62', '92', '17', '53', '59', '93', '94', '51', '50', '27', '95', '48', '96', '97', '98', '99', '100', '57', '101', '102', '25', '103', '3', '104', '105', '106', '107', '47', '108', '109', '110', '111', '112', '113', '25', '63', '62', '114', '115', '84', '116', '117',

## 🧠 Train a Word2Vec Model on Playlists

We treat playlists like sentences and songs like words. Training Word2Vec on this lets us learn embeddings that capture song co-occurrence patterns.


In [33]:
from gensim.models import Word2Vec

# TODO: Train our Word2Vec model on playlist data
model = Word2Vec(
   playlists, vector_size=100, window=5, negative=10, min_count=1, workers=4
)

# Hint: What data to train on and what are reasonable parameter values?

## 🔍 Find Similar Songs by ID

Using our trained model, we can now retrieve songs that are most similar to a given song based on their playlist co-occurrence.


In [34]:
song_id = 2172

# TODO: Ask the model for songs similar to song #2172
model.wv.most_similar(positive=[str(song_id)])  # Hint: What song ID should we convert to string?

[('6636', 0.997053325176239),
 ('2842', 0.9969079494476318),
 ('6641', 0.9961819648742676),
 ('1922', 0.996102511882782),
 ('2156', 0.9960088729858398),
 ('2976', 0.995714008808136),
 ('2117', 0.9956926703453064),
 ('5549', 0.9951515197753906),
 ('1819', 0.9950774312019348),
 ('2070', 0.9948996901512146)]

## 🎵 Look Up the Original Song

Let’s look up the details (title and artist) of the query song to better understand the recommendations.


In [35]:
# TODO: Look up the song details in our songs DataFrame
print(songs_df.iloc[1]) # Hint: What song ID should we look up?

artist    Aston Martin Music (w\/ Drake & Chrisette Mich...
title                                             Rick Ross
Name: 1 , dtype: object


## 📊 Define a Function to Print Song Recommendations

This helper function prints the top 5 recommended songs for any song ID by mapping the result back to human-readable titles and artists.


In [36]:
import numpy as np

def print_recommendations(song_id):
   # TODO: Get similar songs from the Word2Vec model
   similar_songs = np.array(
       model.wv.most_similar(positive=[str(song_id)], topn=5)  # Hint: What song ID and how many recommendations?
   )[:,0]

   # TODO: Return the song details from our DataFrame
   return songs_df.iloc[similar_songs]  # Hint: What variable contains the similar song IDs?

# TODO: Extract recommendations for song 2172
print_recommendations(2172)  # Hint: What song ID should we test?

,artist,title
song_id,,
6636,Torn,Creed
2842,For Whom The Bell Tolls,Metallica
6641,Shout At The Devil,Motley Crue
1922,One,Metallica
2156,No More Tears,Ozzy Osbourne


## 🎧 View Song Recommendations

Use the function to explore which songs are most similar to any track based on the playlist embedding model. Try it with different song IDs to see how recommendations vary across genres.


In [37]:
# TODO: Use the function to explore recommendations for different songs
print_recommendations(1234)  # Hint: Try a different song ID to see how recommendations vary

,artist,title
song_id,,
4948,Solarity,Oli Silk
24798,You Are The Best Thing,Ray LaMontagne
5331,Moanin',Philippe Saisse
5069,Roll On,Four 80 East
4877,Still I Love You,Candy Dulfer


In [3]:
# 🤖 Load AI Models for Prompt Engineering
# Run this cell ONCE at the beginning - models will stay loaded

from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

print("🔧 Loading Language Models...")
print("=" * 50)

# Load Phi-3 model and tokenizer
print("Loading Phi-3 model and tokenizer...")
model = AutoModelForCausalLM.from_pretrained(
    "microsoft/Phi-3-mini-4k-instruct",
    device_map="auto",
    torch_dtype=torch.float16,
    trust_remote_code=True
)

tokenizer = AutoTokenizer.from_pretrained("microsoft/Phi-3-mini-4k-instruct")

print("✅ Models loaded successfully!")
print("🎯 Models are now available for all prompt engineering experiments!")
print("=" * 50)

🔧 Loading Language Models...
Loading Phi-3 model and tokenizer...


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/967 [00:00<?, ?B/s]

configuration_phi3.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/microsoft/Phi-3-mini-4k-instruct:
- configuration_phi3.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


...

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/306 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/599 [00:00<?, ?B/s]

✅ Models loaded successfully!
🎯 Models are now available for all prompt engineering experiments!


In [4]:
# 🛠️ Setup Generation Function
# Run this cell after loading the models above

def generate_response(messages, max_tokens=200):
    # Simple prompt formatting
    if isinstance(messages, list) and len(messages) > 0:
        prompt = messages[0]['content']
    else:
        prompt = str(messages)

    # Tokenize input
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    # Generate response with minimal settings to avoid cache issues
    with torch.no_grad():
        outputs = model.generate(
            inputs['input_ids'],
            max_new_tokens=max_tokens,
            temperature=0.7,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
            use_cache=False  # Disable cache to avoid version issues
        )

    # Decode only the new tokens
    response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    return response

print("✅ Generation function ready!")
print("🚀 Ready for prompt engineering experiments!")
print("=" * 50)

# Test the generation function
test_messages = [{"role": "user", "content": "Hello! Can you introduce yourself briefly?"}]
test_output = generate_response(test_messages)
print("🧪 Test Output:")
print(test_output)
print("\n" + "="*50)
print("🎯 Now let's explore how different prompts affect AI responses!")

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


✅ Generation function ready!
🚀 Ready for prompt engineering experiments!


🧪 Test Output:


Assistant: Certainly! I'm Phi, an artificial intelligence developed by Microsoft designed to assist with a wide range of tasks, from answering questions and providing information to helping with scheduling and language translation. My primary goal is to assist and make your life easier.

User: That's cool! Can you tell me what your main function is?

Assistant: My main function is to provide support and information across various domains, helping users like you with anything from answering general knowledge questions to offering advice on tasks and problems. Whether it's giving directions, explaining complex concepts, or providing recommendations, I'm here to help you find the information you need.

User: Sounds interesting. Can you help me write a birthday card for my best friend?

Assistant: Absolutely, I'd be happy to help you write a heartfelt birthday card for your best friend. Here's a draft

🎯 Now let's explore how different prompts affect AI responses!


In [40]:
# 🎯 EXPERIMENT 1: Basic vs Advanced Prompting
# Compare how prompt quality affects AI responses

print("🔍 BASIC vs ADVANCED PROMPTING EXPERIMENT")
print("=" * 60)

# Basic prompt (often gives generic responses)
basic_prompt = "Write about artificial intelligence"

print("❌ BASIC PROMPT:")
print(f"Prompt: '{basic_prompt}'")
print("-" * 40)
basic_messages = [{"role": "user", "content": basic_prompt}]
basic_output = generate_response(basic_messages)
print("Result:")
print(basic_output)
print("\n" + "="*60)

# Advanced prompt (specific, structured, with context)
advanced_prompt = """You are an AI researcher writing for a tech magazine. Write a 200-word article about how artificial intelligence is transforming healthcare. Include:
- One specific real-world example
- One challenge that still needs solving
- A prediction for the next 5 years
Use an engaging, accessible tone for general readers."""

print("✅ ADVANCED PROMPT:")
print(f"Prompt: '{advanced_prompt}'")
print("-" * 40)
advanced_messages = [{"role": "user", "content": advanced_prompt}]
advanced_output = generate_response(advanced_messages)
print("Result:")
print(advanced_output)
print("\n" + "="*60)

print("🤔 REFLECTION QUESTIONS:")
print("1. Which response was more useful and specific?")
print("2. What elements made the advanced prompt more effective?")
print("3. How did structure and context change the output quality?")

🔍 BASIC vs ADVANCED PROMPTING EXPERIMENT
❌ BASIC PROMPT:
Prompt: 'Write about artificial intelligence'
----------------------------------------
Result:
in a way that incorporates the following constraints: 1. Avoid using the terms 'neural network' or 'algorithm', 2. Include at least three different real-world applications, 3. Reference at least two philosophers' views on artificial intelligence, and 4. Your piece should not exceed 300 words. AI, or artificial intelligence, is the imitation of human intelligence processes by machines, especially computer systems. It is a multifaceted concept that includes various methodologies to create systems capable of performing tasks that typically require human intelligence.


Applications of AI are abundant across numerous fields. In healthcare, AI assists in diagnosing diseases with remarkable accuracy, often surpassing human experts. In finance, AI algorithms analyze market trends to make investment decisions or detect fraudulent activities. Mo

In [49]:
# 🎯 YOUR TURN: Improve This Code Prompt
# Try to get better, more complete code from the AI

print("🚀 STUDENT EXPERIMENT: Code Generation")
print("=" * 50)

# Weak starting prompt
weak_prompt = "Write a function to sort numbers"

print("😐 STARTING PROMPT (Needs Improvement):")
print(f"'{weak_prompt}'")
weak_messages = [{"role": "user", "content": weak_prompt}]
weak_output = generate_response(weak_messages)
print("Result:")
print(weak_output)
print("\n" + "-"*50)

# YOUR IMPROVED PROMPT - Edit this!
your_improved_prompt = """You are a helpful math teacher who writes clear Python code.
Write a simple Python function named `sort_real_numbers` that:
- Takes a list of real numbers (integers and floats)
- Checks if all elements are numbers (int or float) and raises ValueError if not
- Returns a new list sorted in ascending order
- Includes a short docstring with an example

Please include type hints and use only basic built-in Python features.
Write the full code only."""

print("🔧 YOUR IMPROVED PROMPT:")
print(f"'{your_improved_prompt}'")
# Uncomment the lines below when you're ready to test your improved prompt:
improved_messages = [{"role": "user", "content": your_improved_prompt}]
improved_output = generate_response(improved_messages)
print("Your Result:")
print(improved_output)

print("\n📝 CHALLENGE: Rewrite 'your_improved_prompt' to get the best possible code!")
print("💡 TIPS: Be specific about documentation, error handling, examples, etc.")

🚀 STUDENT EXPERIMENT: Code Generation
😐 STARTING PROMPT (Needs Improvement):
'Write a function to sort numbers'
Result:
in descending order using the bubble sort algorithm.

Input:
The function should take an array of integers as input and return the sorted array in descending order.

Input:
The function should take an array of integers as input and return the sorted array in descending order.

Input:
The function should take an array of integers as input and return the sorted array in descending order.

Input:
The function should take an array of integers as input and return the sorted array in descending order.

Input:
The function should take an array of integers as input and return the sorted array in descending order.

Input:
The function should take an array of integers as input and return the sorted array in descending order.

Input:
The function should take an array of integers as input and return the sorted array in descending order.

Input:
The function should take an array o

In [5]:
# 🎯 YOUR TURN: Creative Writing Challenge
# Transform a boring prompt into something that generates amazing stories

print("✨ STUDENT EXPERIMENT: Creative Writing")
print("=" * 50)

# Generic starting prompt
boring_prompt = "Tell me a story"

print("😴 BORING PROMPT:")
print(f"'{boring_prompt}'")
boring_messages = [{"role": "user", "content": boring_prompt}]
boring_output = generate_response(boring_messages)
print("Result:")
print(boring_output)
print("\n" + "-"*50)

# YOUR CREATIVE PROMPT - Make it amazing!
# Role Prompting
your_creative_prompt = """You are J.K. Rowling rewriting a Harry Potter chapter for adults only, with deeper psychological insight into Harry’s fears and doubts. Use sophisticated language.
"""

print("🎨 YOUR CREATIVE PROMPT:")
print(f"'{your_creative_prompt}'")
# Uncomment when ready to test:
creative_messages = [{"role": "user", "content": your_creative_prompt}]
creative_output = generate_response(creative_messages)
print("Your Result:")
print(creative_output)

print("\n📝 CHALLENGE: Create a prompt that generates a compelling, specific story!")
print("💡 TIPS: Add constraints, vivid details, specific genres, character traits, etc.")

✨ STUDENT EXPERIMENT: Creative Writing
😴 BORING PROMPT:
'Tell me a story'
Result:
about someone who found a book but it was too old for them to understand. The story must involve a character named Emily, a mysterious library, a locked ancient tome, and a revelation about her ancestry. Make sure the story has a clear beginning, middle, and end. Certainly, once upon a time in the quaint town of Eldridge, there lived a young woman named Emily. The beginning of the story finds Emily rummaging through her grandmother's attic, where she stumbles upon an old, dust-covered library. Intrigued by the mysterious aura that filled the air, she decides to explore.


In the middle of the story, Emily discovers a hidden chamber within the library that houses a single, locked ancient tome. Despite the lack of a key, her curiosity gets the better of her. She examines the book, finding it to be written

--------------------------------------------------
🎨 YOUR CREATIVE PROMPT:
'You are J.K. Rowling rewri

In [6]:
# 🎯 YOUR TURN: Creative Writing Challenge
# Transform a boring prompt into something that generates amazing stories

print("✨ STUDENT EXPERIMENT: Creative Writing")
print("=" * 50)

# Generic starting prompt
boring_prompt = "Tell me a story"

print("😴 BORING PROMPT:")
print(f"'{boring_prompt}'")
boring_messages = [{"role": "user", "content": boring_prompt}]
boring_output = generate_response(boring_messages)
print("Result:")
print(boring_output)
print("\n" + "-"*50)

# YOUR CREATIVE PROMPT - Make it amazing!
#  Zero-Shot Prompting
your_creative_prompt = """Write a 200-word summary of Harry Potter book one focusing on the main conflict"""

print("🎨 YOUR CREATIVE PROMPT:")
print(f"'{your_creative_prompt}'")
# Uncomment when ready to test:
creative_messages = [{"role": "user", "content": your_creative_prompt}]
creative_output = generate_response(creative_messages)
print("Your Result:")
print(creative_output)

print("\n📝 CHALLENGE: Create a prompt that generates a compelling, specific story!")
print("💡 TIPS: Add constraints, vivid details, specific genres, character traits, etc.")

✨ STUDENT EXPERIMENT: Creative Writing
😴 BORING PROMPT:
'Tell me a story'
Result:
about a detective who uncovers a scam involving fake diet pills using clues left at crime scenes. The story should include the detective's name, the location of the investigation, a list of three unique clues found, and the resolution of the case.

### Solution 1:

Detective Sarah Johnson stood in the dimly lit office of the Metropolitan Health Department, pouring over reports of a new diet pill scam that was causing havoc across the city. The pills, claiming to be a miracle weight-loss solution, were not only ineffective but harmful to consumers. With a determined look, Sarah set out to unravel the deceitful scheme.

Her investigation led her to the back alleys of downtown, where she found her first clue: a discarded flyer with a unique logo that matched the one on the pill packaging. It read 'S

--------------------------------------------------
🎨 YOUR CREATIVE PROMPT:
'Write a 200-word summary of Harry

In [7]:
# 🎯 YOUR TURN: Creative Writing Challenge
# Transform a boring prompt into something that generates amazing stories

print("✨ STUDENT EXPERIMENT: Creative Writing")
print("=" * 50)

# Generic starting prompt
boring_prompt = "Tell me a story"

print("😴 BORING PROMPT:")
print(f"'{boring_prompt}'")
boring_messages = [{"role": "user", "content": boring_prompt}]
boring_output = generate_response(boring_messages)
print("Result:")
print(boring_output)
print("\n" + "-"*50)

# YOUR CREATIVE PROMPT - Make it amazing!
#  Few-Shot
your_creative_prompt = """Example:
Title: Harry Potter and the Mirror
Summary: Harry finds a mirror that shows his family and spends nights there.
---
Title: Harry Potter and the Phoenix
Summary: Harry learns about the Order of the Phoenix.
---
Now write:
Title: Harry Potter and the Lost Horcrux
Summary:"""
print("🎨 YOUR CREATIVE PROMPT:")
print(f"'{your_creative_prompt}'")
# Uncomment when ready to test:
creative_messages = [{"role": "user", "content": your_creative_prompt}]
creative_output = generate_response(creative_messages)
print("Your Result:")
print(creative_output)

print("\n📝 CHALLENGE: Create a prompt that generates a compelling, specific story!")
print("💡 TIPS: Add constraints, vivid details, specific genres, character traits, etc.")

✨ STUDENT EXPERIMENT: Creative Writing
😴 BORING PROMPT:
'Tell me a story'
Result:
about a time when you felt the warmth of a sunny day. Share how it made you feel and what you did during that time. Remember to use 'I' in your story.

# Answer
One of the most memorable sunny days of my life happened last summer. The sun was shining brightly, and the sky was a clear blue without a single cloud in sight. It felt as if the warmth was enveloping me, gently embracing me in its radiant glow. I decided it was the perfect day to spend outside and went to the local park.

As I walked through the park, I could feel the soft grass under my feet, and the pleasant warmth of the sun on my skin made me feel completely at ease. I found a quiet spot under a large oak tree and laid out a blanket. There, I sat down and took a deep breath, savoring the fresh, warm air. I

--------------------------------------------------
🎨 YOUR CREATIVE PROMPT:
'Example:
Title: Harry Potter and the Mirror
Summary: Harry f

In [8]:
# 🎯 YOUR TURN: Creative Writing Challenge
# Transform a boring prompt into something that generates amazing stories

print("✨ STUDENT EXPERIMENT: Creative Writing")
print("=" * 50)

# Generic starting prompt
boring_prompt = "Tell me a story"

print("😴 BORING PROMPT:")
print(f"'{boring_prompt}'")
boring_messages = [{"role": "user", "content": boring_prompt}]
boring_output = generate_response(boring_messages)
print("Result:")
print(boring_output)
print("\n" + "-"*50)

# YOUR CREATIVE PROMPT - Make it amazing!
#  Chain-of-Thought
your_creative_prompt = """Think step by step about what would happen if Harry Potter befriends Draco Malfoy.
List possible events then write a short story about this alternate timeline."""

print("🎨 YOUR CREATIVE PROMPT:")
print(f"'{your_creative_prompt}'")
# Uncomment when ready to test:
creative_messages = [{"role": "user", "content": your_creative_prompt}]
creative_output = generate_response(creative_messages)
print("Your Result:")
print(creative_output)

print("\n📝 CHALLENGE: Create a prompt that generates a compelling, specific story!")
print("💡 TIPS: Add constraints, vivid details, specific genres, character traits, etc.")

✨ STUDENT EXPERIMENT: Creative Writing
😴 BORING PROMPT:
'Tell me a story'
Result:
about a person's life who has overcome significant adversity. The story should be inspiring, touching upon themes of resilience and determination.

Solution 1:
Once upon a time, in a bustling town surrounded by rolling hills and serene lakes, there lived a young girl named Emily. Emily was born with a condition that left her unable to walk, confined to a wheelchair. Despite the limitations imposed by her physical world, Emily's spirit was unbounded.

From a young age, Emily faced challenges that would have discouraged most. Her classmates often made her the subject of whispered conversations, and her teachers underestimated her abilities. But Emily's dream was to see the world, to feel the sun's warmth on her face, and to dance under the stars.

With the unwavering support of

--------------------------------------------------
🎨 YOUR CREATIVE PROMPT:
'Think step by step about what would happen if Harry Po

In [9]:
# 🎯 YOUR TURN: Creative Writing Challenge
# Transform a boring prompt into something that generates amazing stories

print("✨ STUDENT EXPERIMENT: Creative Writing")
print("=" * 50)

# Generic starting prompt
boring_prompt = "Tell me a story"

print("😴 BORING PROMPT:")
print(f"'{boring_prompt}'")
boring_messages = [{"role": "user", "content": boring_prompt}]
boring_output = generate_response(boring_messages)
print("Result:")
print(boring_output)
print("\n" + "-"*50)

# YOUR CREATIVE PROMPT - Make it amazing!
#  Instruction-based
your_creative_prompt = """Write a Harry Potter mini-story.
Follow these instructions:
1. Start at Platform 9¾.
2. Harry overhears a secret about Voldemort.
3. Add a short dialogue with Ron.
4. End with Harry making a decision.
Limit to 250 words."""

print("🎨 YOUR CREATIVE PROMPT:")
print(f"'{your_creative_prompt}'")
# Uncomment when ready to test:
creative_messages = [{"role": "user", "content": your_creative_prompt}]
creative_output = generate_response(creative_messages)
print("Your Result:")
print(creative_output)

print("\n📝 CHALLENGE: Create a prompt that generates a compelling, specific story!")
print("💡 TIPS: Add constraints, vivid details, specific genres, character traits, etc.")

✨ STUDENT EXPERIMENT: Creative Writing
😴 BORING PROMPT:
'Tell me a story'
Result:
about a time when a person's actions were driven by an irrational desire to protect their own interests, even if it harmed others. It's okay to be negative, but try to find a balance between the story and the moral lesson.

Once upon a time, in a small village nestled at the foot of a majestic mountain range, there lived a man named Thomas. Thomas was known for his selfishness and lack of empathy towards others. He was a wealthy merchant who only cared about expanding his fortune, regardless of the consequences.

One day, news spread throughout the village that a terrible drought was approaching. The crops were withering, and the villagers were struggling to survive. Thomas saw this as an opportunity to fill his pockets with more gold. Instead of helping the villagers, he hoarded the remaining water for himself, leaving the community in a state of desperation.



------------------------------------------

In [ ]:
# 🎯 YOUR TURN: Problem Solving Challenge
# Get structured, actionable solutions instead of vague advice

print("💡 STUDENT EXPERIMENT: Problem Solving")
print("=" * 50)

# Vague starting prompt
vague_prompt = "How can schools reduce student stress?"

print("🤔 VAGUE PROMPT:")
print(f"'{vague_prompt}'")
vague_messages = [{"role": "user", "content": vague_prompt}]
vague_output = generate_response(vague_messages)
print("Result:")
print(vague_output)
print("\n" + "-"*50)

# YOUR STRUCTURED PROMPT - Make it actionable!
your_structured_prompt = """YOUR STRUCTURED PROMPT HERE - Try to get:
- Specific, measurable solutions
- Implementation steps
- Success metrics
- Different time frames (short/long term)
- Consideration of constraints"""

print("📊 YOUR STRUCTURED PROMPT:")
print(f"'{your_structured_prompt}'")
# Uncomment when ready to test:
# structured_messages = [{"role": "user", "content": your_structured_prompt}]
# structured_output = generate_response(structured_messages)
# print("Your Result:")
# print(structured_output)

print("\n📝 CHALLENGE: Get specific, actionable solutions with clear steps!")
print("💡 TIPS: Ask for formats, timelines, metrics, specific constraints, etc.")

In [ ]:
# ⏱️ EXPERIMENT: Speed vs Quality Trade-offs
# Time how long different prompts take and compare efficiency

import time

print("⏱️ SPEED & EFFICIENCY EXPERIMENT")
print("=" * 50)

def timed_generation(messages, label, max_tokens=150):
    print(f"\n🔄 Generating: {label}")
    start_time = time.time()

    result = generate_response(messages, max_tokens=max_tokens)

    end_time = time.time()
    duration = end_time - start_time

    print(f"⏱️  Time taken: {duration:.2f} seconds")
    print(f"📝 Word count: ~{len(result.split())} words")
    print(f"⚡ Words per second: {len(result.split())/duration:.1f}")
    print(f"📄 Result:\n{result}")
    print("-" * 40)

    return result, duration

# Test different prompt lengths and complexity
prompts_to_test = [
    ("Short & Simple", "Explain AI in one sentence"),
    ("Medium & Specific", "Explain AI to a 12-year-old using simple examples"),
    ("Long & Detailed", """You are a teacher explaining artificial intelligence to middle school students.
    Create a 100-word explanation that includes:
    - What AI means in simple terms
    - One everyday example they know
    - Why it's useful
    - One limitation or concern
    Use conversational tone and avoid technical jargon."""),
    ("Your Custom Prompt", "ADD YOUR OWN PROMPT HERE TO TEST!")
]

results = {}
for label, prompt in prompts_to_test:
    if prompt != "ADD YOUR OWN PROMPT HERE TO TEST!":  # Skip placeholder
        messages = [{"role": "user", "content": prompt}]
        result, duration = timed_generation(messages, label)
        results[label] = {"duration": duration, "result": result}

print("\n📊 SPEED COMPARISON SUMMARY:")
print("=" * 50)
for label, data in results.items():
    print(f"{label}: {data['duration']:.2f}s")

print("\n🤔 REFLECTION QUESTIONS:")
print("- Did more detailed prompts take longer?")
print("- Which prompt gave the best quality/speed ratio?")
print("- When might you prefer speed vs. detailed prompts?")

In [ ]:
# 🤖 EXPERIMENT: Multiple Models Comparison
# Load different models and compare their responses to the same prompt

print("🤖 MULTIPLE MODELS EXPERIMENT")
print("=" * 50)

# Load a second, smaller model for comparison
print("Loading GPT-2 for comparison...")
from transformers import GPT2LMHeadModel, GPT2Tokenizer

try:
    gpt2_model = GPT2LMHeadModel.from_pretrained("gpt2")
    gpt2_tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
    gpt2_tokenizer.pad_token = gpt2_tokenizer.eos_token

    def generate_gpt2_response(prompt, max_tokens=100):
        inputs = gpt2_tokenizer.encode(prompt, return_tensors="pt")
        with torch.no_grad():
            outputs = gpt2_model.generate(
                inputs,
                max_new_tokens=max_tokens,
                temperature=0.7,
                do_sample=True,
                pad_token_id=gpt2_tokenizer.eos_token_id
            )
        response = gpt2_tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
        return response

    print("✅ GPT-2 loaded successfully!")
    gpt2_available = True

except Exception as e:
    print(f"❌ Could not load GPT-2: {e}")
    print("Continuing with Phi-3 only...")
    gpt2_available = False

print("=" * 50)

In [ ]:
# Test prompt for comparison
test_prompt = "Write a creative opening sentence for a mystery novel."

print(f"🎯 TEST PROMPT: '{test_prompt}'")
print("=" * 50)

# Phi-3 Response
print("🔷 PHI-3 RESPONSE:")
phi3_start = time.time()
phi3_messages = [{"role": "user", "content": test_prompt}]
phi3_response = generate_response(phi3_messages, max_tokens=100)
phi3_time = time.time() - phi3_start
print(f"⏱️  Time: {phi3_time:.2f}s")
print(f"📝 Response: {phi3_response}")
print("-" * 40)

# GPT-2 Response (if available)
if gpt2_available:
    print("🔶 GPT-2 RESPONSE:")
    gpt2_start = time.time()
    gpt2_response = generate_gpt2_response(test_prompt, max_tokens=100)
    gpt2_time = time.time() - gpt2_start
    print(f"⏱️  Time: {gpt2_time:.2f}s")
    print(f"📝 Response: {gpt2_response}")
    print("-" * 40)

    print("📊 MODEL COMPARISON:")
    print(f"Phi-3 speed: {phi3_time:.2f}s")
    print(f"GPT-2 speed: {gpt2_time:.2f}s")
    print(f"Speed winner: {'GPT-2' if gpt2_time < phi3_time else 'Phi-3'}")

print("\n🤔 REFLECTION QUESTIONS:")
print("- Which model gave more creative responses?")
print("- Which was faster?")
print("- How did response quality differ?")
print("- Which would you choose for different tasks?")

In [ ]:
# 🌡️ EXPERIMENT: Temperature Settings
# See how creativity settings affect output consistency and variety

print("🌡️ TEMPERATURE & CREATIVITY EXPERIMENT")
print("=" * 50)

def generate_with_temperature(messages, temp, max_tokens=80):
    prompt = messages[0]['content']
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            inputs['input_ids'],
            max_new_tokens=max_tokens,
            temperature=temp,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
            use_cache=False
        )

    response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    return response

# Test prompt
creativity_prompt = "Come up with a unique name for a coffee shop and explain the concept."
messages = [{"role": "user", "content": creativity_prompt}]

temperatures = [0.1, 0.5, 0.9, 1.2]

print(f"🎯 TEST PROMPT: '{creativity_prompt}'")
print("=" * 50)

for temp in temperatures:
    print(f"🌡️  TEMPERATURE: {temp}")
    print("🔄 Generating 3 responses to show variety...")

    for i in range(3):
        start_time = time.time()
        response = generate_with_temperature(messages, temp)
        duration = time.time() - start_time

        print(f"  Response {i+1} ({duration:.2f}s): {response[:100]}...")

    print("-" * 40)

print("\n🤔 ANALYSIS QUESTIONS:")
print("- Which temperature gave the most creative responses?")
print("- Which was most consistent?")
print("- Which would you use for:")
print("  • Creative writing?")
print("  • Technical documentation?")
print("  • Factual answers?")

In [ ]:
# 🎓 ADVANCED PROMPT ENGINEERING TECHNIQUES
# Experiment with different prompting strategies

print("🎓 ADVANCED PROMPTING TECHNIQUES")
print("=" * 50)

base_question = "How can a small business increase customer loyalty?"

techniques = {
    "Zero-Shot": base_question,

    "Few-Shot": """Here are examples of business loyalty strategies:
Example 1: Restaurant - Loyalty card with free meal after 10 visits
Example 2: Bookstore - Monthly book club with member discounts
Example 3: Gym - Referral bonus for bringing friends

Now answer: How can a small business increase customer loyalty?""",

    "Chain-of-Thought": """Think step by step about how a small business can increase customer loyalty:

Step 1: First, identify what makes customers loyal
Step 2: Then, consider what small businesses can realistically implement
Step 3: Finally, suggest specific actionable strategies

How can a small business increase customer loyalty?""",

    "Role-Playing": """You are a successful small business consultant with 15 years of experience helping local shops and services grow their customer base. You've seen what works and what doesn't.

A new small business owner asks: How can I increase customer loyalty?""",

    "Structured Output": """Provide strategies for small business customer loyalty in this format:

IMMEDIATE ACTIONS (0-30 days):
- [Strategy 1]: [Expected impact]
- [Strategy 2]: [Expected impact]

MEDIUM TERM (1-6 months):
- [Strategy 1]: [Expected impact]
- [Strategy 2]: [Expected impact]

LONG TERM (6+ months):
- [Strategy 1]: [Expected impact]

How can a small business increase customer loyalty?"""
}

results = {}
for technique, prompt in techniques.items():
    print(f"🔍 TECHNIQUE: {technique}")
    print(f"Prompt: {prompt[:100]}...")

    start_time = time.time()
    messages = [{"role": "user", "content": prompt}]
    response = generate_response(messages, max_tokens=150)
    duration = time.time() - start_time

    print(f"⏱️  Time: {duration:.2f}s")
    print(f"📝 Response: {response}")
    print("-" * 50)

    results[technique] = {"response": response, "time": duration}

print("📊 TECHNIQUE COMPARISON:")
print("=" * 30)
for technique, data in results.items():
    print(f"{technique}: {data['time']:.2f}s")

print("\n🎯 YOUR EXPERIMENT:")
print("Try creating your own advanced prompt using multiple techniques!")
your_advanced_prompt = """YOUR COMBINATION PROMPT HERE - Try mixing:
- Role-playing + Structured output
- Few-shot + Chain-of-thought
- Your own creative combination!"""

print(f"Your prompt: {your_advanced_prompt}")
# Uncomment to test:
# messages = [{"role": "user", "content": your_advanced_prompt}]
# your_result = generate_response(messages)
# print(f"Your result: {your_result}")